# 01 — CHEMBL206 Verisini Alma, Sütunları Tanıma ve SMILES Kontrolü

**Hedef:** `CHEMBL206_IC50_single_protein_format_CLEAN_parent_dedup.csv` dosyasını Google Drive'dan okumak, kritik sütunları otomatik belirlemek, pIC50 değerini doğrulamak/üretmek ve sonraki notebooklar için standart bir başlangıç dosyası hazırlamak.

### Bu notebookta öğreneceklerimiz
- Google Drive bağlantısı kurma
- CSV dosyasını güvenli biçimde okuma
- SMILES ve aktivite sütunlarını otomatik bulma
- `IC50 → pIC50` dönüşümünü kontrol etme
- Veri kalite özeti ve checkpoint kullanımı

### Ders planındaki yeri
**Ders 1 / Bölüm A:** Veri yapısını tanıma ve çalışma klasörünü hazırlama.

## Workshop dosya yapısı

Bu seri çalıştırıldığında Google Drive altında aşağıdaki yapı oluşur:

```text
MyDrive/
└── CHEMBL206_workshop/
    ├── 01_data/
    ├── 02_cleaned/
    ├── 03_features/
    ├── 04_models/
    └── 05_reports/
```

> **Çalıştırma sırası:** Notebook 01 → 02 → 03 → 04 → 05  
> Her kod hücresinin sonunda bir **CHECKPOINT** mesajı vardır. Bir hücre hata verirse sonraki hücreye geçmeyiniz.

## Bilimsel not: IC50 ve pIC50

IC50 molar birime çevrildikten sonra:

\[
pIC_{50} = -\log_{10}(IC_{50}\,[M])
\]

Örnek: `IC50 = 100 nM = 1×10⁻⁷ M` ise `pIC50 = 7`.

Bu veri seti yalnızca **IC50 + single protein format** kayıtlarından geldiği için, dosyada bulunan `pStandard` sütunu varsa bu notebook onu `pIC50` olarak adlandırır. Yine de ham değer ve birim mevcutsa dönüşümü ayrıca kontrol eder.

In [ ]:
# Google Colab oturumunda Google Drive'ı bağlamak için gerekli modülü içe aktarırız.
from google.colab import drive

# Drive bağlantısını /content/drive klasörüne kurarız.
# Daha önce bağlandıysa Colab bunu ekranda belirtecektir.
drive.mount("/content/drive")

print("✅ CHECKPOINT 01.1: Google Drive bağlantısı kuruldu.")

In [ ]:
# Dosya yollarını yönetmek için pathlib kullanıyoruz.
from pathlib import Path

# Tablo işlemleri için pandas kullanıyoruz.
import pandas as pd

# Sayısal hesaplamalar için numpy kullanıyoruz.
import numpy as np

# Notebook içinde tabloları daha okunur göstermek için display kullanıyoruz.
from IPython.display import display

# Workshop ana klasörünü tanımlıyoruz.
PROJECT_ROOT = Path("/content/drive/MyDrive/CHEMBL206_workshop")

# Kullanıcının ekrandaki klasör yapısına göre kaynak CSV yolunu tanımlıyoruz.
SOURCE_CSV = Path(
    "/content/drive/MyDrive/CHEMBL206_cleaned_sets/"
    "CHEMBL206_IC50_single_protein_format_CLEAN_parent_dedup.csv"
)

# Notebook 01 çıktılarının kaydedileceği klasörü tanımlıyoruz.
DATA_DIR = PROJECT_ROOT / "01_data"

# Gerekli klasörleri yoksa oluşturuyoruz.
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Kaynak dosyanın gerçekten var olup olmadığını kontrol ediyoruz.
if not SOURCE_CSV.exists():
    raise FileNotFoundError(
        "Kaynak CSV bulunamadı. SOURCE_CSV değişkenindeki yolu "
        "Google Drive klasör yapınıza göre güncelleyiniz:\n"
        f"{SOURCE_CSV}"
    )

print("Kaynak CSV:", SOURCE_CSV)
print("Çıktı klasörü:", DATA_DIR)
print("✅ CHECKPOINT 01.2: Dosya yolları hazır ve kaynak CSV bulundu.")

In [ ]:
# CSV dosyasını pandas DataFrame olarak okuyoruz.
df_raw = pd.read_csv(SOURCE_CSV, low_memory=False)

# Veri kümesinin satır ve sütun sayısını yazdırıyoruz.
print("Veri boyutu (satır, sütun):", df_raw.shape)

# İlk 10 satırı ekranda gösteriyoruz.
display(df_raw.head(10))

# Sütun adlarını numaralı biçimde yazdırıyoruz.
print("\nSütunlar:")
for i, column_name in enumerate(df_raw.columns, start=1):
    print(f"{i:02d}. {column_name}")

# Boş bir veri setiyle devam edilmesini engelliyoruz.
assert len(df_raw) > 0, "CSV dosyası boş görünüyor."

print("✅ CHECKPOINT 01.3: CSV başarıyla okundu ve temel görünüm kontrol edildi.")

In [ ]:
# Aday sütun adlarını büyük/küçük harf farkından bağımsız arayan yardımcı fonksiyonu tanımlıyoruz.
def find_column(dataframe, candidates, required=True):
    # Gerçek sütun adlarını küçük harfli bir sözlükte eşliyoruz.
    lower_map = {str(col).lower(): col for col in dataframe.columns}

    # Adayları verilen sırayla kontrol ediyoruz.
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    # Zorunlu sütun bulunamazsa açıklayıcı hata veriyoruz.
    if required:
        raise KeyError(
            "Beklenen sütun bulunamadı. Aranan adaylar: "
            + ", ".join(candidates)
        )

    # Zorunlu değilse None döndürüyoruz.
    return None


# SMILES için olası sütun adlarını tanımlıyoruz.
SMILES_CANDIDATES = [
    "canonical_smiles",
    "parent_smiles",
    "smiles",
    "molecule_smiles",
    "canonical_smile",
]

# Molekül kimliği için olası sütun adlarını tanımlıyoruz.
ID_CANDIDATES = [
    "molecule_chembl_id",
    "molecule_id",
    "chembl_id",
    "compound_id",
]

# Aktivitenin logaritmik hali için olası sütun adlarını tanımlıyoruz.
PACTIVITY_CANDIDATES = [
    "pIC50",
    "pStandard",
    "pstandard",
    "pchembl_value",
    "pChEMBL",
]

# Ham aktivite değeri ve birimi için olası sütunları tanımlıyoruz.
VALUE_CANDIDATES = ["standard_value", "standard_value_num", "IC50"]
UNIT_CANDIDATES = ["standard_units", "units", "unit"]

# Sütunları otomatik seçiyoruz.
smiles_col = find_column(df_raw, SMILES_CANDIDATES, required=True)
id_col = find_column(df_raw, ID_CANDIDATES, required=False)
pactivity_col = find_column(df_raw, PACTIVITY_CANDIDATES, required=False)
value_col = find_column(df_raw, VALUE_CANDIDATES, required=False)
unit_col = find_column(df_raw, UNIT_CANDIDATES, required=False)

print("Seçilen SMILES sütunu:", smiles_col)
print("Seçilen molekül ID sütunu:", id_col)
print("Seçilen pIC50/pStandard sütunu:", pactivity_col)
print("Seçilen ham IC50 sütunu:", value_col)
print("Seçilen birim sütunu:", unit_col)

assert smiles_col is not None, "SMILES sütunu belirlenemedi."

print("✅ CHECKPOINT 01.4: Kritik sütunlar otomatik olarak tanımlandı.")

In [ ]:
# IC50 birimlerini molar birime çevirmek için dönüşüm katsayılarını tanımlıyoruz.
UNIT_TO_MOLAR = {
    "M": 1.0,
    "mM": 1e-3,
    "uM": 1e-6,
    "µM": 1e-6,
    "μM": 1e-6,
    "nM": 1e-9,
    "pM": 1e-12,
}


# Ham IC50 ve birim sütunlarından pIC50 hesaplayan fonksiyonu tanımlıyoruz.
def calculate_pic50(values, units):
    # Ham değerleri sayısal formata çeviriyoruz; hatalı girişleri NaN yapıyoruz.
    numeric_values = pd.to_numeric(values, errors="coerce")

    # Birimleri temiz bir metin dizisine çeviriyoruz.
    clean_units = units.astype(str).str.strip()

    # Her birim için molar dönüşüm katsayısını eşliyoruz.
    factors = clean_units.map(UNIT_TO_MOLAR)

    # IC50 değerlerini molar birime çeviriyoruz.
    ic50_molar = numeric_values * factors

    # Sıfır veya negatif değerleri fiziksel olarak geçersiz kabul ediyoruz.
    ic50_molar = ic50_molar.where(ic50_molar > 0)

    # pIC50 = -log10(IC50[M]) formülünü uyguluyoruz.
    return -np.log10(ic50_molar)


# Orijinal tabloyu korumak için kopya alıyoruz.
df_base = df_raw.copy()

# SMILES sütununu standart bir isim altında kopyalıyoruz.
df_base["smiles_raw"] = df_base[smiles_col].astype("string").str.strip()

# Molekül kimliği varsa standart isimle kopyalıyoruz.
if id_col is not None:
    df_base["molecule_id"] = df_base[id_col].astype("string")
else:
    # Molekül kimliği yoksa satır numarasından geçici bir kimlik üretiyoruz.
    df_base["molecule_id"] = [f"ROW_{i:06d}" for i in range(len(df_base))]

# Dosyada pIC50/pStandard/pChEMBL benzeri bir sütun varsa onu kullanıyoruz.
if pactivity_col is not None:
    df_base["pIC50"] = pd.to_numeric(df_base[pactivity_col], errors="coerce")
    pactivity_source = pactivity_col

# Logaritmik aktivite sütunu yoksa ham IC50 ve birimden hesaplıyoruz.
elif value_col is not None and unit_col is not None:
    df_base["pIC50"] = calculate_pic50(df_base[value_col], df_base[unit_col])
    pactivity_source = f"{value_col} + {unit_col}"

# Hiçbir uygun aktivite bilgisi yoksa açıklayıcı hata veriyoruz.
else:
    raise KeyError(
        "pIC50 üretilemedi. pStandard/pIC50/pChEMBL veya "
        "standard_value + standard_units sütunları gereklidir."
    )

print("pIC50 kaynağı:", pactivity_source)
print(df_base[["molecule_id", "smiles_raw", "pIC50"]].head(10))

print("✅ CHECKPOINT 01.5: SMILES ve pIC50 standart sütunları oluşturuldu.")

In [ ]:
# Eksik SMILES sayısını hesaplıyoruz.
missing_smiles = int(
    df_base["smiles_raw"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

# Eksik pIC50 sayısını hesaplıyoruz.
missing_pic50 = int(df_base["pIC50"].isna().sum())

# Tam tekrar eden ham SMILES sayısını hesaplıyoruz.
duplicate_raw_smiles = int(df_base["smiles_raw"].duplicated(keep=False).sum())

# pIC50 dağılımının özetini hazırlıyoruz.
pic50_summary = df_base["pIC50"].describe()

print("Eksik SMILES:", missing_smiles)
print("Eksik pIC50:", missing_pic50)
print("Tekrar grubunda bulunan ham SMILES satırı:", duplicate_raw_smiles)
print("\npIC50 özeti:")
display(pic50_summary.to_frame("pIC50"))

# İlk 20 SMILES'ı ders sırasında kolayca incelemek için yazdırıyoruz.
print("\nİlk 20 SMILES:")
for row_number, smiles in enumerate(df_base["smiles_raw"].head(20), start=1):
    print(f"{row_number:02d}: {smiles}")

print("✅ CHECKPOINT 01.6: Veri kalite özeti ve SMILES ön izlemesi tamamlandı.")

In [ ]:
# Sonraki notebooklara aktarılacak temel sütunları öne alıyoruz.
front_columns = ["molecule_id", "smiles_raw", "pIC50"]

# Diğer orijinal sütunları bilgi kaybı olmadan sona ekliyoruz.
remaining_columns = [
    col for col in df_base.columns
    if col not in front_columns
]

# Sütun sırasını düzenliyoruz.
df_base = df_base[front_columns + remaining_columns]

# SMILES veya pIC50 değeri eksik satırları modelleme başlangıç dosyasından çıkarıyoruz.
# Bu işlem kimyasal standardizasyon değildir; yalnızca zorunlu alan kontrolüdür.
df_workshop_base = df_base.dropna(subset=["smiles_raw", "pIC50"]).copy()

# Boş metin SMILES'ları ayrıca çıkarıyoruz.
df_workshop_base = df_workshop_base[
    df_workshop_base["smiles_raw"].astype(str).str.len() > 0
].copy()

# Ana çıktının yolunu tanımlıyoruz.
BASE_OUTPUT = DATA_DIR / "CHEMBL206_IC50_workshop_base.csv"

# Kalite kontrol özetinin yolunu tanımlıyoruz.
QC_OUTPUT = DATA_DIR / "CHEMBL206_IC50_notebook01_QC.csv"

# Temel çalışma tablosunu CSV olarak kaydediyoruz.
df_workshop_base.to_csv(BASE_OUTPUT, index=False)

# Basit kalite kontrol tablosunu hazırlıyoruz.
qc_table = pd.DataFrame(
    {
        "metric": [
            "input_rows",
            "output_rows",
            "missing_smiles",
            "missing_pIC50",
            "raw_duplicate_smiles_rows",
            "pIC50_min",
            "pIC50_max",
        ],
        "value": [
            len(df_base),
            len(df_workshop_base),
            missing_smiles,
            missing_pic50,
            duplicate_raw_smiles,
            df_workshop_base["pIC50"].min(),
            df_workshop_base["pIC50"].max(),
        ],
    }
)

# Kalite kontrol tablosunu kaydediyoruz.
qc_table.to_csv(QC_OUTPUT, index=False)

print("Kaydedilen temel veri:", BASE_OUTPUT)
print("Kaydedilen kalite raporu:", QC_OUTPUT)
display(qc_table)

# Çıktı dosyasının gerçekten oluştuğunu kontrol ediyoruz.
assert BASE_OUTPUT.exists(), "Temel veri CSV'si oluşturulamadı."
assert len(df_workshop_base) > 0, "Temel veri dosyasında hiç satır kalmadı."

print("✅ CHECKPOINT 01.7: Notebook 01 çıktıları başarıyla kaydedildi.")

## Ders sonu değerlendirme soruları

1. SMILES sütunu neden farklı veri tabanlarında farklı isimlerle bulunabilir?
2. `pIC50` değerinin büyümesi daha güçlü mü, daha zayıf mı aktiviteyi gösterir?
3. Aynı molekülün birden fazla aktivite ölçümü varsa doğrudan bir satırı silmek neden risklidir?
4. Bu aşamada tuz temizliği neden yapılmadı?

**Sonraki notebook:** Tuz/fragman temizliği, parent SMILES üretimi, tekrarların birleştirilmesi ve Lipinski filtresi.